In [ ]:

# Remember to run necessary cells from previous questions before running this cell.
# The functions below are given by the tutorials. See 8.2.1_RSA and 8.2.3 for information.
import sys, os
sys.path.append(os.getcwd())

from cryptography_functions import gcd, random_prime
from random import SystemRandom, randint
def char_to_byte(char):

    byte = bin(ord(char))[2:]
    zeroes = 8 - len(byte)
    byte = '0' * zeroes + byte

    return byte

def integer(char, verbose=False):
    binary = '1'
    binary = binary + char_to_byte(char)
    if verbose:
        return(binary)
    return int(binary, 2)

def RSA_key_generator(bitlength=128):

# The output is ((PublicKey), PrivateExp, Factorisation).

    ## Generate two large primes.
    p = random_prime(bitlength)
    q = random_prime(bitlength)
    N = p*q
    
    ## Next, get phi and use it to get a public and private exponent.
    phi = (p-1)*(q-1)
    ## Choose encryption exponent e randomly between 1 and p and q.
    e = randint(1,min(p,q))
    ## It must be coprime to phi though. We use rejection sampling again:    
    while gcd(e,phi) > 1:
        e = randint(1,min(p,q))
    
    ## Finally, get d using 'pow'
    ## (the extended Euclid's algorithm could do this too).
    d = pow(e,-1,phi)
    
    ## Then return the keys.
    return ((N,e),d%phi,{p:1,q:1})


# My code begins here.


def key_slice(key):

    # First we turn the key into a list of characters and find how many steps we need to do.
    num_key = []
    list_key = [str(i) for i in str(key)]
    for j in list_key:
        num = integer(j)
        num_ten = [str(i) for i in str(num)]
        num_key.extend(num_ten)
        
    step = len(num_key) // 36
    sliced = []
    
    # Null case for if we do not need to slice
    
    if step == 0:
        ''.join(str(l) for l in num_key)
        return [''.join(num_key)]

    # We split every seven digits, until we reach a remainder, then we split the rest. Each time we do this, we join the list back into a string and add it to a list.
    
    else:  
        for n in range(1, step + 2):
            if n<= step + 1:
                x = slice(36*(n-1), 36*n)
                if num_key[x] != []:
                    sliced.append(''.join(str(m) for m in num_key[x]))
        
        return sliced

def sliced_vigenere(txt):

    # I built the code below using the vigenere test in order to generate my key.
    
    plaintext = get_plaintext(txt)
    key_len = random.randint(1, len(plaintext))
    key = ''
    for i in range(key_len):
        key += ascii_lowercase[random.randint(0, 25)]

    # We apply the cipher and slice it ready for RSA encryption.
        
    cipher_text = vigenere_cipher(plaintext, key)
    vigenere_slice = key_slice(key)
    return vigenere_slice, cipher_text

def rsa_encrypt(txt, pub):

    # We assign variable names to our vigenere cipher and encrypted text
    
    rsa_vigenere = []
    s, cipher_txt = sliced_vigenere(txt)

    # We assign RSA encryption to each slice, and combine the slices back into one key
    
    for t in range(len(s)):
        t = pow(int(s[t]),pub[1],pub[0])
        rsa_vigenere.append(t)
    
    return rsa_vigenere, cipher_txt

def decrypt(key, cipher_txt, pub, priv):

    # We undo the rsa encryption with the private key.
    
    decrypted = []
    for i in txt:
        decrypted.append(str(pow(i, priv, pub[0])))

    # Because we add a 1 to the beginning of the binary representations of each character (see lecture 8.2.3), every letter is represented by a 3 digit number.
    # Therefore we can split our key into groups of three, subtract 256 and apply chr to get our vigenere key.
    # Finally we apply the decryption to decipher the message.

    # One thing to note is that spaces are lost somewhere, maybe in the vigenere cipher?
        
    key = ''
    group = ''.join(decrypted)
    
    for j in range(0, len(group), 3):
        x = slice(j, j+3)
        key = key + chr(int(group[x]) - 256)
        
    raw = decrypt_vigenere_cipher(cipher_txt, key)
    print(raw)
test = 'The Quick Brown Fox Jumps Over The Lazy Dog'  

In [ ]:
# Text cells below
test = 'The Quick Brown Fox Jumps Over The Lazy Dog' # For if you want to use something less blinding to test.

pub, priv, fact = RSA_key_generator()

key, text = rsa_encrypt(test, pub)

In [ ]:
decrypt(key, text, pub, priv)


In [ ]:
pub, priv, fact = RSA_key_generator()

# Test 1: do the keys work on a small number?
m = 123456
c = pow(m, pub[1], pub[0])
print(pow(c, priv, pub[0]) == m)      # should print True

# Test 2: are the chunks 36 digits and starting with 3?
for chunk in key_slice('abcdefghijklmnopqrstuvwxyz'):
    print(len(chunk), chunk)

# Test 3: what does decrypt actually produce?
enc, cipher_txt = rsa_encrypt('test.txt', pub)
print(pub, priv)